# Non-Instruction Causal LLM Fine-Tuning on Bihar PDF

## Domain-Adaptive Continued Pretraining on a Complex Document

This notebook performs **non-instruction causal language model fine-tuning** on the Bihar.pdf document.

### What makes this different from a simple text PDF?
Bihar.pdf contains:
- Dense academic text (history, economics, governance)
- Tables with statistical data
- Figures and charts
- Footnotes and references

### Our approach:
1. We extract **text** (the primary training signal for causal LM)
2. We extract **tables** and convert them to text representations
3. We note images/figures but do NOT train on them (causal LM = text only)
4. We combine all textual content into a training corpus

### Pipeline
```text
Bihar PDF (complex document)
   ↓
Text extraction (PyMuPDF page-level)
   ↓
Table extraction (pdfplumber → text representation)
   ↓
Text cleaning and normalization
   ↓
Paragraph splitting + deduplication
   ↓
Hugging Face Dataset creation
   ↓
Tokenization + text packing into fixed blocks
   ↓
LoRA/QLoRA fine-tuning (TinyLlama)
   ↓
Validation loss + perplexity
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

### Why Non-Instruction Fine-Tuning?
We feed the model **raw domain text** and it learns to predict the next token.
The model learns:
- Bihar-specific terminology (Magadha, Pataliputra, GSDP, bifurcation)
- Economic and governance language
- Historical writing style
- Statistical patterns and table descriptions

The model does **NOT** learn:
- How to answer questions
- How to follow instructions
- How to be a chatbot

(That would require Instruction Fine-Tuning in a separate step)

---
## Step 1: Install Required Libraries

We install:
- `pymupdf`: For PDF text and image extraction
- `pdfplumber`: For table extraction (works without Ghostscript unlike camelot)
- `datasets`: Hugging Face dataset creation
- `transformers`, `accelerate`: Model loading, tokenizer, Trainer
- `peft`: LoRA/QLoRA adapters
- `bitsandbytes`: 4-bit quantized model loading
- `sentencepiece`: Required by some tokenizers

In [11]:
!pip install -q -U "numpy<2.1" "scipy" "pymupdf" "datasets" "transformers" "accelerate" "peft" "bitsandbytes" "sentencepiece" "pillow<11.0" "pymupdf4llm" "docling" "pandas" "opencv-python" "tqdm" "pdfplumber"

---
## Step 2: Import All Libraries

We organize imports by category for clarity.

In [12]:
# ============================================================
# Standard Libraries
# ============================================================
import os
import re
import gc
import json
import math
import random
import warnings
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

# ============================================================
# Data Handling
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# PDF Processing
# ============================================================
import fitz          # PyMuPDF - for text + image extraction
import pdfplumber    # For table extraction

# ============================================================
# Hugging Face
# ============================================================
from datasets import Dataset, DatasetDict

# ============================================================
# PyTorch
# ============================================================
import torch

# ============================================================
# Transformers
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ============================================================
# PEFT (LoRA / QLoRA)
# ============================================================
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("All imports successful!")

All imports successful!


---
## Step 3: Global Configuration

We put ALL parameters in a single dataclass.

**Why?**
- Easy to change one value and re-run the entire notebook
- Reproducibility: you can save/share the config
- No magic numbers scattered across cells

In [13]:
@dataclass
class Config:
    # ===== DATA =====
    # Upload Bihar.pdf to Colab and set this path
    pdf_path: str = "/content/Bihar.pdf"

    # ===== MODEL =====
    # TinyLlama: lightweight, good for learning/demo
    # For production, use Llama-2-7B, Mistral-7B, etc.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # ===== OUTPUT DIRECTORIES =====
    output_dir: str = "/content/bihar_lora_output"
    adapter_dir: str = "/content/bihar_lora_adapter"
    processed_data_dir: str = "/content/bihar_processed_data"

    # ===== TEXT PREPROCESSING =====
    # Paragraphs shorter than this are discarded (noise)
    min_chars_per_paragraph: int = 100
    # Each training block will have this many tokens
    block_size: int = 512

    # ===== TRAIN/EVAL SPLIT =====
    test_size: float = 0.10
    seed: int = 42

    # ===== LoRA PARAMETERS =====
    lora_r: int = 16           # Rank of LoRA matrices
    lora_alpha: int = 32       # Scaling factor
    lora_dropout: float = 0.05 # Dropout for regularization

    # ===== TRAINING PARAMETERS =====
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_steps: int = 10
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 20
    save_steps: int = 50
    save_total_limit: int = 2
    # Set max_steps=30 for quick demo, -1 for full training
    max_steps: int = -1


config = Config()
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Bihar.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/bihar_lora_output",
  "adapter_dir": "/content/bihar_lora_adapter",
  "processed_data_dir": "/content/bihar_processed_data",
  "min_chars_per_paragraph": 100,
  "block_size": 512,
  "test_size": 0.1,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_steps": 10,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "eval_steps": 20,
  "save_steps": 50,
  "save_total_limit": 2,
  "max_steps": -1
}


In [14]:
config = Config()
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)
print("Directories created.")

Directories created.


In [15]:
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF not found at: /content/Bihar.pdf


---
## Step 5: Extract Text from PDF (Page-Level)

PyMuPDF (`fitz`) extracts all text content from each page.

Bihar.pdf has 94 pages with:
- Academic paragraphs
- Footnotes
- Table captions and data
- Figure references

We extract everything as raw text first, then clean it.

In [16]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from each page of a PDF using PyMuPDF.

    Returns a list of dicts with:
    - page: page number (1-indexed)
    - text: raw extracted text
    - char_count: length of text
    """
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [19]:
pdf_pages = extract_pdf_pages(config.pdf_path)

print(f"Total pages with extracted text: {len(pdf_pages)}")
print(f"\nPage-level character counts:")
for p in pdf_pages[:10]:  # Show first 10 pages
    print(f"  Page {p['page']:3d}: {p['char_count']:,} characters")
if len(pdf_pages) > 10:
    print(f"  ... and {len(pdf_pages) - 10} more pages")

total_chars = sum(p['char_count'] for p in pdf_pages)
print(f"\nTotal raw characters: {total_chars:,}")

Total pages with extracted text: 94

Page-level character counts:
  Page   1: 236 characters
  Page   2: 3,657 characters
  Page   3: 3,901 characters
  Page   4: 4,158 characters
  Page   5: 4,077 characters
  Page   6: 4,107 characters
  Page   7: 4,056 characters
  Page   8: 3,814 characters
  Page   9: 3,916 characters
  Page  10: 4,094 characters
  ... and 84 more pages

Total raw characters: 275,335


In [20]:
# Preview first page raw text
print("=" * 80)
print("RAW TEXT - Page 1 (first 1500 chars):")
print("=" * 80)
print(pdf_pages[0]["text"][:1500])

RAW TEXT - Page 1 (first 1500 chars):
1 
 
Bihar: What Went Wrong? And What Changed? 
 
Arnab Mukherji and Anjan Mukherji 
 
 
 
Working Paper No. 2012-107 
 
September 2012 
 
 
 
 
 
 
 
 
National Institute of Public Finance and Policy 
New Delhi 
http://www.nipfp.org.in


---
## Step 6: Extract Tables from PDF

Bihar.pdf has statistical tables (poverty data, HDI, crime rates, etc.).

We use `pdfplumber` to extract tables and convert them to text.

**Why convert tables to text?**
- Causal LM training requires text sequences
- The model can learn patterns like: "In 1981, Bihar's per capita income was Rs. 917"
- Table data converted to readable text adds domain knowledge

In [21]:
def extract_tables_as_text(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract tables from PDF using pdfplumber and convert to text.

    Each table is converted to a string representation that can be
    included in the training corpus.
    """
    tables_data = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables()
            for t_index, table in enumerate(page_tables):
                if not table or len(table) < 2:
                    continue

                # Convert to DataFrame for clean text representation
                try:
                    # First row as header
                    headers = [str(h).strip() if h else f"Col_{i}"
                              for i, h in enumerate(table[0])]
                    df = pd.DataFrame(table[1:], columns=headers)

                    # Convert to readable text
                    table_text = df.to_string(index=False)

                    if len(table_text) > 50:  # Skip tiny/empty tables
                        tables_data.append({
                            "page": page_index,
                            "table_index": t_index + 1,
                            "rows": len(df),
                            "cols": len(df.columns),
                            "text": table_text,
                            "char_count": len(table_text),
                        })
                except Exception as e:
                    # Skip malformed tables
                    continue

    return tables_data

In [22]:
extracted_tables = extract_tables_as_text(config.pdf_path)

print(f"Total tables extracted: {len(extracted_tables)}")
print()

# Show first 3 tables
for i, t in enumerate(extracted_tables[:3]):
    print(f"Table {i+1} | Page {t['page']} | {t['rows']} rows x {t['cols']} cols")
    print(t['text'][:300])
    print("-" * 60)

Total tables extracted: 25

Table 1 | Page 68 | 6 rows x 18 cols
0.65\n0.55\n0.45\n0.35\n0.25\n1981 1983 1985 1987 1989 1991 1993 1995 1997 1999 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10       Col_11 Col_12 Col_13 05 2007 2009 2011 Col_15 Col_16 Col_17
                                                                           None              
------------------------------------------------------------
Table 2 | Page 69 | 8 rows x 27 cols
Col_0 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10 Col_11 Col_12 Col_13 Col_14 Col_15 Col_16 Col_17 Col_18 Col_19 Col_20 Col_21 Col_22 Col_23 Col_24 Col_25 Col_26
                                                               NaN                  NaN                         NaN      
------------------------------------------------------------
Table 3 | Page 69 | 15 rows x 26 cols
Col_0 Col_1 Col_2 Col_3 Col_4 Col_5 Col_6 Col_7 Col_8 Col_9 Col_10 Col_11 Col_12 Col_13 Col_14 Col_15 Col_16 Col_17 Col_18 Col_19 Col_20

---
## Step 7: Extract Image/Figure Metadata (For Reference Only)

**Important:** We do NOT train on images for causal LM.

But we log how many images exist for documentation purposes.
If you later want to do multimodal training, you would need a different approach.

In [23]:
def count_images(pdf_path: str) -> Dict[str, Any]:
    """Count embedded images in the PDF for documentation."""
    image_count = 0
    pages_with_images = []

    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            images = page.get_images(full=True)
            if images:
                image_count += len(images)
                pages_with_images.append(page_index)

    return {
        "total_images": image_count,
        "pages_with_images": pages_with_images,
    }

image_info = count_images(config.pdf_path)
print(f"Total embedded images/figures: {image_info['total_images']}")
print(f"Pages containing images: {image_info['pages_with_images'][:20]}")
print()
print("NOTE: Images are NOT used for text-based causal LM training.")
print("The model learns from text only. Figure captions (if present in text) ARE included.")

Total embedded images/figures: 36
Pages containing images: [51, 52, 54, 68, 71, 72, 73, 74, 76, 77, 78, 79, 82]

NOTE: Images are NOT used for text-based causal LM training.
The model learns from text only. Figure captions (if present in text) ARE included.


---
## Step 8: Text Cleaning

PDF extraction produces messy text. We need to clean it.

| Problem | Solution |
|---------|----------|
| Unicode oddities (ﬁ, ﬂ) | `unicodedata.normalize("NFKC")` |
| Zero-width spaces | Remove them |
| Hyphenated line breaks (gluconeogene-\nsis) | Join the word |
| Multiple spaces/tabs | Single space |
| Standalone page numbers | Remove |
| Excessive blank lines | Normalize to \n\n |

In [24]:
def clean_pdf_text(text: str) -> str:
    """
    Clean raw PDF-extracted text while preserving meaningful content.

    This handles common PDF extraction artifacts:
    - Unicode normalization
    - Hidden characters
    - Hyphenated line breaks
    - Multiple whitespace
    - Page numbers
    """
    # 1. Normalize Unicode (e.g., ﬁ → fi, ＡＭＰＫ → AMPK)
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove zero-width and BOM characters
    text = text.replace("\u200b", "")  # zero-width space
    text = text.replace("\ufeff", "")  # byte order mark

    # 3. Fix hyphenated line breaks: "gover-\nnance" → "governance"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # 4. Normalize multiple spaces/tabs to single space
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 6. Remove standalone page numbers (lines with only digits)
    text = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", text)

    # 7. Split into paragraphs, clean each, reassemble
    paragraphs = re.split(r"\n\s*\n", text)
    cleaned_paragraphs = []

    for para in paragraphs:
        # Remove internal line breaks (PDF wrapping)
        para = re.sub(r"\n+", " ", para)
        # Normalize spaces
        para = re.sub(r"\s+", " ", para).strip()
        if para:
            cleaned_paragraphs.append(para)

    return "\n\n".join(cleaned_paragraphs)

In [25]:
# Clean all extracted pages
cleaned_pages = []
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print(f"Cleaned {len(cleaned_pages)} pages.")
print(f"Total cleaned characters: {sum(p['char_count'] for p in cleaned_pages):,}")
print()
print("Preview of cleaned Page 3:")
print("=" * 80)
print(cleaned_pages[2]["text"][:1200])

Cleaned 94 pages.
Total cleaned characters: 265,657

Preview of cleaned Page 3:
“For decades the sprawling state of Bihar, flat and scorching as a griddle, was something between a punch line and a cautionary tale, ... Criminals could count on the police for protection, not prosecution. Highwaymen ruled the shredded roads and kidnapping was one of the state’s most profitable businesses... Its government, led by politicians who used divisive identity politics to entrench their rule, was so corrupt that it required a newly coined phrase: the Jungle Raj.” Polgreen (2010)

This is an idea of Bihar that the majority of contemporary readers have encountered over and over again, especially after the 1980s. Polgreen (2010), however, in this article, goes on to describe not the decay of a once successful nation state, but rather a more remarkable change--the veritable signs of development and growth in a state that had once been considered a basket-case, or more politely, a failed state (see Fig

---
## Step 9: Split Into Paragraph Records

We split the cleaned text into individual paragraphs because:
- Easier to inspect and audit
- Can filter by length (remove noise)
- Can deduplicate repeated content
- Later, tokenization will pack these into fixed-size blocks anyway

In [26]:
def split_into_paragraph_records(
    cleaned_pages: List[Dict[str, Any]],
    min_chars: int = 100
) -> List[Dict[str, Any]]:
    """
    Split cleaned pages into individual paragraph records.

    - Filters out short paragraphs (likely noise/headers)
    - Deduplicates exact matches
    - Tracks source page for auditability
    """
    records = []
    seen = set()  # For deduplication

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])

        for para_idx, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()

            # Skip short paragraphs (noise, headers, page numbers)
            if len(paragraph) < min_chars:
                continue

            # Deduplicate
            key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if key in seen:
                continue
            seen.add(key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": para_idx,
                "char_count": len(paragraph),
            })

    return records

In [27]:
# Create paragraph records from page text
paragraph_records = split_into_paragraph_records(
    cleaned_pages,
    min_chars=config.min_chars_per_paragraph
)

print(f"Total paragraph records from page text: {len(paragraph_records)}")
print()

# Also add table text as paragraph records
table_records = []
for t in extracted_tables:
    if t["char_count"] >= config.min_chars_per_paragraph:
        table_records.append({
            "text": f"Table data from page {t['page']}:\n{t['text']}",
            "source_page": t["page"],
            "paragraph_id": 0,  # 0 indicates table source
            "char_count": t["char_count"],
        })

print(f"Table records added: {len(table_records)}")

# Combine all records
all_records = paragraph_records + table_records
print(f"\nTotal training records: {len(all_records)}")
print(f"Total characters in corpus: {sum(r['char_count'] for r in all_records):,}")

Total paragraph records from page text: 418

Table records added: 25

Total training records: 443
Total characters in corpus: 290,640


In [28]:
# Preview some records
print("Sample records:")
for i in [0, 5, len(all_records)//2]:
    if i < len(all_records):
        print("=" * 80)
        print(f"Record {i} | Page {all_records[i]['source_page']} | {all_records[i]['char_count']} chars")
        print(all_records[i]["text"][:300])
        print()

Sample records:
Record 0 | Page 2 | 717 chars
Bihar as a political entity, either as a kingdom, or as a state within the republic of India, has its own identity from the time written records were available (Thapar 1966; Rangarajan 1992). Noted historian, Romila Thapar, describes the history of ancient India as the history of ancient Bihar. Many

Record 5 | Page 3 | 1013 chars
The Bihar section of the Essays on State Policies is organized in the following way: in Chapter 1 we provide a historical narrative of Bihar to provide context to much of its current state, and then we focus on its contemporary economy in the past three decades to better understand the moribund stat

Record 221 | Page 50 | 2010 chars
The Nitish Kumar-led government, whether by serendipity or design, appears to have achieved some success and can be expected to continue to deliver on these aspects with by now excellent experience of mobilizing and delivering public services. To keep himself aware of the concerns and

---
## Step 10: Save Processed Data (Auditability)

Always save intermediate data in real projects.
This helps with debugging, reproducibility, and compliance.

In [29]:
# Save processed corpus
corpus_path = os.path.join(config.processed_data_dir, "bihar_corpus.jsonl")

with open(corpus_path, "w", encoding="utf-8") as f:
    for record in all_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(all_records)} records to: {corpus_path}")

Saved 443 records to: /content/bihar_processed_data/bihar_corpus.jsonl


---
## Step 11: Create Hugging Face Dataset + Train/Eval Split

We convert our records into a Hugging Face `Dataset` object.

Then we split into train and validation sets.

**Why keep a validation set?**
- Gives us validation loss (tells us if model is learning)
- Can compute perplexity (lower = better language understanding)
- Detects overfitting

In [30]:
if len(all_records) < 5:
    raise ValueError(
        "Corpus too small! Need at least 5 records. "
        "Check if PDF path is correct and extraction worked."
    )

# Create HF Dataset
text_dataset = Dataset.from_list(all_records)
print(f"Dataset: {text_dataset}")
print(f"\nSample entry:")
print(text_dataset[0])

Dataset: Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 443
})

Sample entry:
{'text': 'Bihar as a political entity, either as a kingdom, or as a state within the republic of India, has its own identity from the time written records were available (Thapar 1966; Rangarajan 1992). Noted historian, Romila Thapar, describes the history of ancient India as the history of ancient Bihar. Many achievements that India became renowned for, in education, governance, society, or religion, have their roots in Bihar. Significant achievements of Bihar in trade and economic engagement within the state and outside of the Indian sub-continent emerge from a past that appears to have left no living legacy in today’s Bihar--a past so alien as to be either simply forgotten or treated as being completely incredible.4', 'source_page': 2, 'paragraph_id': 5, 'char_count': 717}


In [31]:
# Train/validation split
split_dataset = text_dataset.train_test_split(
    test_size=config.test_size,
    seed=config.seed
)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)
print(f"\nTrain samples: {len(raw_datasets['train'])}")
print(f"Validation samples: {len(raw_datasets['validation'])}")

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 398
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 45
    })
})

Train samples: 398
Validation samples: 45


---
## Step 12: Load Tokenizer

The tokenizer converts text → token IDs.

For causal LM:
```
"Bihar's economy" → [1, 350, 28742, 29915, 29879, 29871, 25378, 7586]
```

The model then learns: given tokens 1-7, predict token 8.

In [32]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some models don't have a pad token - set it to EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer: {config.model_name}")
print(f"Vocab size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32,000
Pad token: '</s>' (id=2)
EOS token: '</s>' (id=2)


---
## Step 13: Tokenize + Pack Into Fixed Blocks

Two-step process:
1. **Tokenize**: Convert all text to token IDs
2. **Pack/Group**: Concatenate all tokens, then split into blocks of `block_size` (512)

**Why pack?**
- Without packing: short paragraphs need lots of padding → wasted compute
- With packing: all blocks are exactly 512 tokens → efficient training

```
Para 1 (100 tokens) + Para 2 (200 tokens) + Para 3 (212 tokens) = Block 1 (512 tokens)
Para 4 (300 tokens) + Para 5 (212 tokens) = Block 2 (512 tokens)
```

In [33]:
def tokenize_function(examples):
    """Tokenize text without padding (packing handles it later)."""
    return tokenizer(examples["text"])


def group_texts(examples):
    """
    Pack tokenized sequences into fixed-size blocks.

    Steps:
    1. Concatenate all token sequences into one long sequence
    2. Split into chunks of block_size
    3. Drop the last incomplete chunk
    4. Labels = input_ids (model shifts internally for next-token prediction)
    """
    # Concatenate all sequences
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])

    # Drop remainder
    total_length = (total_length // config.block_size) * config.block_size

    if total_length == 0:
        return {k: [] for k in concatenated.keys()}

    # Split into blocks
    result = {
        k: [t[i:i + config.block_size] for i in range(0, total_length, config.block_size)]
        for k, t in concatenated.items()
    }

    # For causal LM: labels = input_ids
    result["labels"] = result["input_ids"].copy()
    return result

In [34]:
# Step 1: Tokenize
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing",
)

print("After tokenization:")
print(tokenized_datasets)

Tokenizing:   0%|          | 0/398 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/45 [00:00<?, ? examples/s]

After tokenization:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 398
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 45
    })
})


In [35]:
# Step 2: Pack into fixed blocks
final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing into {config.block_size}-token blocks",
)

print("\nAfter packing:")
print(final_datasets)
print(f"\nTrain blocks: {len(final_datasets['train'])}")
print(f"Validation blocks: {len(final_datasets['validation'])}")

if len(final_datasets['train']) == 0:
    raise ValueError("No training blocks created! PDF might be too small or block_size too large.")

Packing into 512-token blocks:   0%|          | 0/398 [00:00<?, ? examples/s]

Packing into 512-token blocks:   0%|          | 0/45 [00:00<?, ? examples/s]


After packing:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 141
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12
    })
})

Train blocks: 141
Validation blocks: 12


In [36]:
# Verify a sample block
sample = final_datasets["train"][0]
print(f"Sample block - input_ids length: {len(sample['input_ids'])}")
print(f"Sample block - labels length: {len(sample['labels'])}")
print(f"\nDecoded preview (first 300 chars):")
print(tokenizer.decode(sample["input_ids"][:100]))

Sample block - input_ids length: 512
Sample block - labels length: 512

Decoded preview (first 300 chars):
<s> [2000-01, 2009-10] [2005-06,2009-10] Agriculture -0.20% 3.60% Industry 2.00% 6.00% Construction 23.70% 23.20% Services 8.40% 11.90% Source: Central Statistical Organ


---
## Step 14: Load Base Model with QLoRA Configuration

We load TinyLlama in **4-bit quantized** format:
- Uses ~1/4 the GPU memory
- Enables fine-tuning on free Colab GPUs (T4, 15GB)
- Industry standard for parameter-efficient fine-tuning

If CUDA is not available, we fall back to float32 (very slow on CPU).

In [37]:
use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")
if use_cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [38]:
# Clear memory before loading
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    # 4-bit quantization config (QLoRA)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4 quantization
        bnb_4bit_compute_dtype=torch.float16, # Compute in FP16
        bnb_4bit_use_double_quant=True,       # Double quantization saves more memory
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Required for stable gradients with quantized models
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable KV cache during training (saves memory, avoids warnings)
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


---
## Step 15: Apply LoRA Adapters

**LoRA (Low-Rank Adaptation):**
- Instead of updating ALL 1.1B parameters, we add small trainable matrices
- Only ~1-3% of parameters are trained
- Much faster and cheaper than full fine-tuning
- The adapter can be saved separately (~30-50MB vs 2GB+ for full model)

**Target modules:**
- `q_proj, k_proj, v_proj, o_proj`: Attention layers
- `gate_proj, up_proj, down_proj`: Feed-forward layers

In [39]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---
## Step 16: Data Collator

The data collator prepares mini-batches during training.

- `mlm=False` → We are doing **causal** LM (predict next token), NOT masked LM (BERT-style)

In [40]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

---
## Step 17: Training Arguments

These control how training proceeds.

In [41]:
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
    seed=config.seed,
)

print("Training arguments configured.")

Training arguments configured.


---
## Step 18: Build Trainer and Start Training

In [42]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer ready. Starting training...")
print(f"Training blocks: {len(final_datasets['train'])}")
print(f"Validation blocks: {len(final_datasets['validation'])}")

Trainer ready. Starting training...
Training blocks: 141
Validation blocks: 12


In [43]:
# ============================================================
# TRAIN!
# ============================================================
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print(f"Train loss: {train_result.training_loss:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
20,2.127438,2.071847
40,2.028132,1.974688
54,1.916823,1.970258



TRAINING COMPLETE!
Train loss: 2.1312


---
## Step 19: Evaluate - Validation Loss & Perplexity

**Perplexity** = how "surprised" the model is by the validation text.
- Lower perplexity = model understands the domain better
- Formula: `perplexity = exp(validation_loss)`

In [44]:
eval_results = trainer.evaluate()

eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print()
print("Interpretation:")
print(f"  The model is, on average, choosing from ~{perplexity:.0f} equally likely next tokens.")
print(f"  Lower is better. Typical values for domain FT: 5-50.")

Training Loss,Validation Loss,Step
1.916823,1.970258,54


Validation Loss: 1.9703
Perplexity: 7.17

Interpretation:
  The model is, on average, choosing from ~7 equally likely next tokens.
  Lower is better. Typical values for domain FT: 5-50.


---
## Step 20: Save LoRA Adapter

We save only the adapter (small, ~30-50MB), not the full model.

To use it later:
1. Load the base model (TinyLlama)
2. Load the adapter on top
3. Generate text

In [45]:
# Save adapter + tokenizer
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print(f"\nSaved files:")
for f in os.listdir(config.adapter_dir):
    size = os.path.getsize(os.path.join(config.adapter_dir, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

LoRA adapter saved to: /content/bihar_lora_adapter

Saved files:
  README.md (5.1 KB)
  tokenizer.json (3534.1 KB)
  adapter_model.safetensors (49319.9 KB)
  tokenizer_config.json (0.4 KB)
  adapter_config.json (1.1 KB)


---
## Step 21: Reload Model + Adapter for Inference

This simulates how you would use the model in production:
1. Load fresh base model
2. Load saved LoRA adapter
3. Generate domain-specific continuations

In [46]:
# Clean up training objects
del trainer, model, base_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

print("Memory cleared. Reloading for inference...")

Memory cleared. Reloading for inference...


In [47]:
# Reload base model
if use_cuda:
    reload_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    inference_base = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=reload_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Load tokenizer from adapter directory
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

# Load LoRA adapter on top of base model
inference_model = PeftModel.from_pretrained(inference_base, config.adapter_dir)
inference_model.eval()

print("Model + adapter loaded for inference!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model + adapter loaded for inference!


---
## Step 22: Text Continuation Inference

Since this is **non-instruction** fine-tuning, we give the model a text prompt
and it **continues** writing in the style/domain it learned.

Good prompts look like the beginning of a paragraph from Bihar.pdf.

**Bad prompt** (instruction-style): "What happened in Bihar after 2005?"

**Good prompt** (continuation-style): "After the 2005 elections in Bihar, Nitish Kumar"

In [50]:
def generate_continuation(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate text continuation from a prompt.

    Parameters:
    - prompt: Starting text (should look like document text)
    - max_new_tokens: How many tokens to generate

    Returns:
    - Full text (prompt + generated continuation)
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,         # Sampling for diverse outputs
            temperature=0.7,        # Lower = more focused
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1, # Reduce repetition
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [51]:
# ============================================================
# TEST: Domain-specific text continuations
# ============================================================

test_prompts = [
    "Bihar as a political entity has its own identity from the time",
    "The Permanent Settlement Act of 1793 had devastating consequences for",
    "After the 2005 elections, the Nitish Kumar government focused on",
    "The bifurcation of Bihar into Bihar and Jharkhand in 2000 resulted in",
    "Bihar's per capita income relative to the national average",
]

for prompt in test_prompts:
    print("=" * 90)
    print(f"PROMPT: {prompt}")
    print("-" * 90)
    result = generate_continuation(prompt, max_new_tokens=150)
    # Show only the generated part
    continuation = result[len(prompt):]
    print(f"CONTINUATION: {continuation}")
    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Bihar as a political entity has its own identity from the time
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  of its formation, it was not at all an easy task to bring about the changes necessary for economic growth and development. The first thing that needs to be emphasized is that in 1972, the government of Bihar had an annual revenue of Rs. 3,600 crore and a net capital receipt of Rs. 855 crore; this was a much smaller state than what it was even before the separation of Bihar from India, with a per capita income of Rs. 438, which meant that Bihar could only afford to spend on public services a fraction of what was available to the Central Government. This was clearly not sustainable, and the

PROMPT: The Permanent Settlement Act of 1793 had devastating consequences for
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  the Indian economy, and even today, much of India's poverty remains structural in nature. While there have been several attempts to study and quantify the impact of this act on the economy, a more comprehensive approach is needed. The authors discuss two such approaches: the Rao-Rao (2008) approach that uses data from the Permanent Settlement Report to study the short-run effects of the Act, while the other approach (Guruswamy, Gautam, and Bapat (2010)) makes use of a long-run model to estimate the long-run effects of the Act. They argue that these are complementary in capturing different aspects of the impact of

PROMPT: After the 2005 elections, the Nitish Kumar government focused on
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  strengthening state institutions and made efforts to tackle corruption. It also continued with reforms in the education system and tried to improve the quality of public services. The government was able to enforce law and order and control crime. In fact, crime rates dropped significantly, with the Crime Against Women (Cognizance) Rules being introduced in 2008. However, the economic situation remained poor and the economy was unable to recover. There were several factors that led to this; these included a lack of infrastructure, weak agricultural productivity, and low levels of industrialization. These constraints are still present today. The state’s fiscal position has improved considerably over the past few years but it remains one

PROMPT: The bifurcation of Bihar into Bihar and Jharkhand in 2000 resulted in
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTINUATION:  substantial changes in the state’s economy, but at the same time it also created a huge set of challenges for governance. A number of policy changes were made to respond to these challenges; however, there is still much room for improvement and a large amount of work remains to be done to make the state government more efficient and effective. Становништво Mukherjee (2014) argues that while Bihar was poor before the bifurcation, its per capita income after bifurcation was not much different from that of the national average. This led to a large disparity between the growth rates achieved by Bihar and the rest of India, even though both states were on similar economic pathways before

PROMPT: Bihar's per capita income relative to the national average
------------------------------------------------------------------------------------------
CONTINUATION: , it is one of the poorest states in India. Over the past decade, Bihar has lost over 30% of its human capital and nearl

---
## Step 23: Merge Stage 1 LoRA Adapter Into Base Model

We merge the trained LoRA adapter into the base model to create a standalone model.

**Why merge?**
- Creates a single model file without needing separate adapter loading
- Required before adding a NEW LoRA adapter for Stage 2 (instruction tuning)
- The merged model becomes the "new base" for subsequent fine-tuning stages

```text
Base TinyLlama + Stage 1 LoRA adapter
   ↓ merge_and_unload()
Standalone domain-adapted TinyLlama (Bihar knowledge baked in)
```

In [52]:
# Clean up inference objects before merging
del inference_model, inference_base, inference_tokenizer
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

print("Memory cleared for merge step.")

Memory cleared for merge step.


In [54]:
# Update torchao to resolve version mismatch error
!pip install -q -U torchao

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
import os

# Reload base model in float16 for safe merging
merge_base = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None,
    trust_remote_code=True,
)

# Load Stage 1 LoRA adapter on top
merge_model = PeftModel.from_pretrained(merge_base, config.adapter_dir)

# Merge adapter weights into base model weights
merged_model = merge_model.merge_and_unload()

# Save merged model
merged_model_dir = "/content/bihar_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

merged_model.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)

print(f"Merged Stage 1 model saved to: {merged_model_dir}")
print(f"\nSaved files:")
for f in os.listdir(merged_model_dir):
    size = os.path.getsize(os.path.join(merged_model_dir, f)) / (1024 * 1024)
    print(f"  {f} ({size:.1f} MB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged Stage 1 model saved to: /content/bihar_merged_model

Saved files:
  tokenizer.json (3.5 MB)
  model.safetensors (2098.2 MB)
  tokenizer_config.json (0.0 MB)
  generation_config.json (0.0 MB)
  config.json (0.0 MB)


In [55]:
# Clean up merge objects
del merge_base, merge_model, merged_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

print("Stage 1 complete. Ready for Stage 2: Instruction Fine-Tuning.")

Stage 1 complete. Ready for Stage 2: Instruction Fine-Tuning.


---
---

# Stage 2: Instruction Fine-Tuning on Bihar Domain-Adapted Model

## Continue from Stage 1 Merged Model → Instruction-Following Capability

In Stage 1, we performed **non-instruction fine-tuning** (domain-adaptive continued pretraining) on Bihar.pdf.

Now we continue from the **merged Stage 1 model** and perform **instruction fine-tuning** using structured Bihar-domain instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw Bihar text continued pretraining (LoRA)
   ↓
Merged domain-adapted model (Bihar knowledge baked in)
   ↓
Stage 2: Instruction fine-tuning on Bihar Q&A data (new LoRA)
   ↓
Final instruction-tuned Bihar LoRA adapter
```

### What changes in Instruction Fine-Tuning?

| Aspect | Stage 1 (Non-Instruction) | Stage 2 (Instruction) |
|--------|---------------------------|----------------------|
| Data format | Raw text paragraphs | Instruction → Response pairs |
| Model learns | Domain language, terminology | How to answer questions |
| Prompt style | Text continuation | "Explain...", "What is..." |
| Output style | Continues writing | Structured answers |

### Example of Instruction Data

```json
{
  "instruction": "Explain the significance of the bifurcation of Bihar in 2000.",
  "input": "",
  "output": "The bifurcation of Bihar into Bihar and Jharkhand in November 2000 resulted in Bihar losing about 45% of its geographical area and significant mineral resources..."
}
```

---
## Step 24: Pipeline for Stage 2

```text
Non-Instruction FT (Stage 1 - RAW Bihar text)
      ↓
Merged Stage 1 model saved
      ↓
Load merged Stage 1 model
      ↓
Perform Instruction FT on same model (Bihar Q&A data)
      ↓
Save instruction-tuned LoRA adapter
      ↓
Load and test with instruction-style prompts
      ↓
Merge instruction adapter for Stage 3
```

We are training a **new LoRA adapter** on top of the merged Stage 1 model.

---
## Step 25: Create Bihar Instruction Dataset

Since Bihar.pdf is an academic document, we create instruction-response pairs
that reflect the kind of questions a user might ask about Bihar.

In a real project, you would:
1. Generate these from the PDF using GPT-4 / Claude
2. Have domain experts write them manually
3. Use a combination of both

For this demo, we create a synthetic instruction dataset programmatically.

In [56]:
# ============================================================
# Create Bihar instruction dataset
# ============================================================
# In production, generate these from the PDF using an LLM or have
# domain experts create them. Here we use a synthetic dataset.

bihar_instruction_data = [
    {
        "instruction": "Explain the historical significance of Bihar as a political entity.",
        "input": "",
        "output": "Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region has historically been a center of political power, learning, and cultural development in the Indian subcontinent. The ancient university of Nalanda was located in Bihar, making it a hub of intellectual activity for centuries."
    },
    {
        "instruction": "What was the impact of the Permanent Settlement Act of 1793 on Bihar?",
        "input": "",
        "output": "The Permanent Settlement Act of 1793 had devastating consequences for Bihar's rural population. It created a class of zamindars who became intermediaries between the British government and the cultivators. The peasants were subjected to excessive rent extraction, leading to widespread poverty and indebtedness. The system concentrated land ownership in the hands of a few while impoverishing the majority of agricultural workers."
    },
    {
        "instruction": "Describe the consequences of Bihar's bifurcation in 2000.",
        "input": "",
        "output": "The bifurcation of Bihar into Bihar and Jharkhand in November 2000 resulted in Bihar losing approximately 45% of its geographical area along with significant mineral and forest resources that went to Jharkhand. Bihar retained the more densely populated plains but lost its industrial base. The state's revenue declined sharply, and it had to rebuild its economic foundations with a primarily agrarian economy and limited mineral resources."
    },
    {
        "instruction": "What reforms did the Nitish Kumar government implement after 2005?",
        "input": "",
        "output": "After the 2005 elections, the Nitish Kumar government focused on governance reforms including improved law and order, road construction, education expansion, and women's empowerment. The government implemented the Bihar Special Courts Act, expanded the road network significantly, reserved 50% of panchayat seats for women, and introduced bicycle distribution schemes for girl students. These reforms led to a period of accelerated economic growth often called Bihar's turnaround."
    },
    {
        "instruction": "Explain Bihar's economic performance relative to the national average.",
        "input": "",
        "output": "Bihar's per capita income has historically been significantly below the national average. After bifurcation in 2000, the gap widened further as the state lost its mineral-rich southern region. However, from 2005 onwards, Bihar experienced higher-than-national-average GDP growth rates, though the absolute per capita income gap remained substantial due to the low base. The state's GSDP growth averaged over 10% annually during 2005-2015, among the highest in India."
    },
    {
        "instruction": "What are the major challenges facing Bihar's education system?",
        "input": "",
        "output": "Bihar's education system faces multiple challenges including a high student-teacher ratio, inadequate infrastructure in government schools, low female literacy rates particularly in rural areas, high dropout rates at the secondary level, and insufficient technical and vocational training institutions. Despite improvements in enrollment through programs like Mukhyamantri Balika Cycle Yojana and midday meal schemes, learning outcomes remain below national benchmarks."
    },
    {
        "instruction": "Describe the agricultural economy of Bihar.",
        "input": "",
        "output": "Agriculture is the backbone of Bihar's economy, employing over 70% of the workforce. The state is a major producer of rice, wheat, maize, lentils, and vegetables. The fertile Gangetic plains provide ideal conditions for cultivation, but the sector suffers from fragmented landholdings, frequent floods, limited irrigation coverage, and poor market linkages. The average farm size in Bihar is among the smallest in India, making mechanization difficult."
    },
    {
        "instruction": "What role did the Magadha Empire play in Indian history?",
        "input": "",
        "output": "The Magadha Empire, centered in present-day Bihar with its capital at Pataliputra (modern Patna), was one of the most powerful kingdoms in ancient India. It gave rise to the Maurya dynasty under Chandragupta Maurya and reached its zenith under Emperor Ashoka. Magadha was also the birthplace of Buddhism and Jainism, and the Gupta Empire that followed ushered in India's Golden Age of science, mathematics, and art."
    },
    {
        "instruction": "Explain the flood problem in Bihar and its economic impact.",
        "input": "",
        "output": "Bihar is one of the most flood-prone states in India, with about 73% of its area susceptible to flooding from rivers originating in Nepal. The Kosi, Gandak, and Bagmati rivers cause recurring devastation. Floods destroy crops, livestock, and infrastructure annually, displacing millions of people. The economic cost includes loss of agricultural output, damage to roads and buildings, and long-term setbacks to development in affected districts."
    },
    {
        "instruction": "What is the significance of Pataliputra in Bihar's history?",
        "input": "",
        "output": "Pataliputra, modern-day Patna, served as the capital of successive empires including the Magadha, Nanda, Maurya, and Gupta dynasties. Founded around the 5th century BCE, it was one of the largest cities in the ancient world. Greek ambassador Megasthenes described it as a magnificent city. Pataliputra was a center of trade, learning, and administration for over a millennium, making it one of the longest-serving capital cities in world history."
    },
    {
        "instruction": "How has Bihar's Human Development Index changed over time?",
        "input": "",
        "output": "Bihar has consistently ranked among the lowest Indian states on the Human Development Index. Key indicators including life expectancy, literacy rate, and per capita income have been below national averages. However, post-2005 reforms led to measurable improvements in education enrollment, infant mortality rates, and poverty reduction. The HDI improved from 0.367 in 2001 to approximately 0.447 by 2011, though it remained among the lowest nationally."
    },
    {
        "instruction": "Describe the caste dynamics and social structure in Bihar.",
        "input": "",
        "output": "Bihar's social structure has been heavily influenced by the caste system. The state has a complex caste hierarchy that has historically determined land ownership, political power, and access to education. Land reforms in the post-independence period attempted to address inequalities but had limited success. Caste-based mobilization has been a defining feature of Bihar's politics, with parties often organized along caste lines and reservation policies playing a significant role in social change."
    },
    {
        "instruction": "What industrial development has occurred in Bihar?",
        "input": "",
        "output": "Bihar's industrial development has been limited compared to other Indian states. After losing the mineral-rich Jharkhand region in 2000, the state had to focus on agro-based industries, food processing, and services. Recent initiatives include special economic zones, industrial area development authorities, and incentives for investment. The state has seen growth in the dairy industry, sugar mills, and IT services, but manufacturing remains underdeveloped relative to its population."
    },
    {
        "instruction": "Explain the role of migration in Bihar's economy.",
        "input": "",
        "output": "Bihar is one of the largest sources of internal migration in India. Millions of workers migrate to states like Maharashtra, Delhi, Punjab, and Gujarat for employment in construction, manufacturing, and services. Remittances from migrant workers form a significant portion of household income in Bihar. While migration provides economic relief, it also reflects the lack of local employment opportunities and contributes to brain drain from the state."
    },
    {
        "instruction": "What transportation challenges does Bihar face?",
        "input": "",
        "output": "Bihar faces significant transportation challenges including inadequate road connectivity in rural areas, limited railway network coverage relative to population, frequent disruption of road and rail links due to flooding, and insufficient bridges across major rivers. The Ganga river divides the state, and until recently only a few bridges connected north and south Bihar. Road quality has improved substantially since 2005 but still lags behind national standards in many districts."
    },
    {
        "instruction": "Describe Bihar's poverty statistics and trends.",
        "input": "",
        "output": "Bihar has historically had one of the highest poverty rates in India. According to the Tendulkar methodology, about 53.5% of Bihar's population lived below the poverty line in 2004-05. This declined to approximately 33.7% by 2011-12, representing significant improvement but still remaining well above the national average of 21.9%. Rural poverty has been particularly acute, driven by landlessness, low agricultural productivity, and limited non-farm employment opportunities."
    },
    {
        "instruction": "What healthcare challenges exist in Bihar?",
        "input": "",
        "output": "Bihar's healthcare system faces severe challenges including shortage of medical professionals, inadequate primary health centers in rural areas, high infant and maternal mortality rates, poor sanitation infrastructure, and limited access to specialist care. The state has one of the lowest doctor-to-population ratios in India. Despite improvements in immunization coverage and institutional delivery rates, health outcomes remain significantly below national averages."
    },
    {
        "instruction": "Explain the governance reforms that improved Bihar's law and order after 2005.",
        "input": "",
        "output": "After 2005, the Bihar government implemented several governance reforms to address the severe law and order problems. These included fast-track courts for speedy justice, strengthening of police forces, crackdown on criminal-political nexus, seizure of illegally held arms, and the Bihar Special Courts Act. The government also focused on dismantling private armies and reducing kidnapping-for-ransom incidents. These measures significantly improved the investment climate and public confidence in governance."
    },
]

print(f"Created {len(bihar_instruction_data)} instruction-response pairs for Bihar domain.")
print(f"\nSample entry:")
print(json.dumps(bihar_instruction_data[0], indent=2))

Created 18 instruction-response pairs for Bihar domain.

Sample entry:
{
  "instruction": "Explain the historical significance of Bihar as a political entity.",
  "input": "",
  "output": "Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region has historically been a center of political power, learning, and cultural development in the Indian subcontinent. The ancient university of Nalanda was located in Bihar, making it a hub of intellectual activity for centuries."
}


In [57]:
# Save instruction data to JSONL for auditability
instruction_data_path = os.path.join(config.processed_data_dir, "bihar_instruction_dataset.jsonl")

with open(instruction_data_path, "w", encoding="utf-8") as f:
    for record in bihar_instruction_data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Instruction dataset saved to: {instruction_data_path}")

Instruction dataset saved to: /content/bihar_processed_data/bihar_instruction_dataset.jsonl


---
## Step 26: Format Instruction Records (Alpaca Style)

We convert every record into **Alpaca-style** training text:

```text
### Instruction:
{instruction}

### Input:
{input}  (if provided)

### Response:
{output}
```

This format teaches the model:
1. To recognize when it receives an instruction
2. To generate a response after `### Response:`
3. To use domain knowledge (from Stage 1) in its answers

In [58]:
def format_instruction_record(record: Dict[str, str]) -> Dict[str, str]:
    """
    Convert an instruction record into Alpaca-style formatted text.

    Parameters:
    - record: dict with 'instruction', 'input' (optional), 'output'

    Returns:
    - dict with 'text' key containing formatted training text
    """
    instruction = record.get("instruction", "").strip()
    input_text = record.get("input", "").strip()
    output_text = record.get("output", "").strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [59]:
# Format all instruction records
formatted_instruction_data = [format_instruction_record(r) for r in bihar_instruction_data]

print(f"Formatted {len(formatted_instruction_data)} instruction records.")
print()
print("Sample formatted text:")
print("=" * 80)
print(formatted_instruction_data[0]["text"])

Formatted 18 instruction records.

Sample formatted text:
### Instruction:
Explain the historical significance of Bihar as a political entity.

### Response:
Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region has historically been a center of political power, learning, and cultural development in the Indian subcontinent. The ancient university of Nalanda was located in Bihar, making it a hub of intellectual activity for centuries.


---
## Step 27: Create HF Dataset + Train/Eval Split for Instructions

In [60]:
# Create Hugging Face Dataset from formatted instruction data
instruction_dataset = Dataset.from_list(formatted_instruction_data)
print(f"Instruction dataset: {instruction_dataset}")
print(f"\nSample entry:")
print(instruction_dataset[0])

Instruction dataset: Dataset({
    features: ['text'],
    num_rows: 18
})

Sample entry:
{'text': '### Instruction:\nExplain the historical significance of Bihar as a political entity.\n\n### Response:\nBihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region has historically been a center of political power, learning, and cultural development in the Indian subcontinent. The ancient university of Nalanda was located in Bihar, making it a hub of intellectual activity for centuries.'}


In [61]:
# Train/validation split
instruction_split = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=config.seed
)

instruction_datasets = DatasetDict({
    "train": instruction_split["train"],
    "validation": instruction_split["test"],
})

print(instruction_datasets)
print(f"\nTrain examples: {len(instruction_datasets['train'])}")
print(f"Validation examples: {len(instruction_datasets['validation'])}")

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3
    })
})

Train examples: 15
Validation examples: 3


---
## Step 28: Tokenize Instruction Dataset

For instruction fine-tuning, we use **padding + truncation** (not text packing).

**Why not text packing here?**
- Each instruction-response pair is a complete training example
- We don't want to concatenate unrelated Q&A pairs
- Padding tokens are ignored in loss using `-100` labels

| Token Position | Content | Label |
|---------------|---------|-------|
| 1-100 | Real instruction + response tokens | Real token IDs |
| 101-512 | Padding tokens (PAD) | -100 (ignored in loss) |

In [62]:
# Reload tokenizer (same as Stage 1)
instruction_tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if instruction_tokenizer.pad_token is None:
    instruction_tokenizer.pad_token = instruction_tokenizer.eos_token

instruction_tokenizer.padding_side = "right"

INSTRUCTION_MAX_LENGTH = 512

print(f"Tokenizer: {config.model_name}")
print(f"Max length for instruction tokenization: {INSTRUCTION_MAX_LENGTH}")

Tokenizer: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Max length for instruction tokenization: 512


In [63]:
def tokenize_instruction_function(examples):
    """
    Tokenize instruction text with padding and truncation.

    Labels are set to -100 for padding tokens so the model
    does not learn to predict padding.
    """
    tokens = instruction_tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=INSTRUCTION_MAX_LENGTH,
    )

    # For causal LM, labels = input_ids
    tokens["labels"] = tokens["input_ids"].copy()

    # Mask padding tokens in labels with -100 (ignored by CrossEntropyLoss)
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [64]:
# Tokenize instruction dataset
instruction_tokenized = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print("Tokenized instruction dataset:")
print(instruction_tokenized)
print(f"\nSample - input_ids length: {len(instruction_tokenized['train'][0]['input_ids'])}")
print(f"Sample - labels length: {len(instruction_tokenized['train'][0]['labels'])}")

Tokenizing instruction dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenized instruction dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3
    })
})

Sample - input_ids length: 512
Sample - labels length: 512


---
## Step 29: Load Merged Stage 1 Model + New LoRA Adapter

We load the **merged Stage 1 model** (which has Bihar domain knowledge baked in)
and attach a **fresh LoRA adapter** for instruction tuning.

```text
Merged Stage 1 model (Bihar domain knowledge)
   + New LoRA adapter (learns instruction-following)
   = Stage 2 trainable model
```

In [65]:
# Clear memory
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Load merged Stage 1 model
if use_cuda:
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)
else:
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

print("Merged Stage 1 model loaded for instruction fine-tuning.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Merged Stage 1 model loaded for instruction fine-tuning.


In [66]:
# Create a new LoRA adapter for instruction fine-tuning
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

instruction_model = get_peft_model(instruction_base_model, instruction_lora_config)
instruction_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---
## Step 30: Data Collator + Training Arguments for Instruction FT

In [67]:
# Data collator for instruction fine-tuning
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=instruction_tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

# Output directories for instruction fine-tuning
instruction_output_dir = "/content/bihar_instruction_lora_output"
instruction_adapter_dir = "/content/bihar_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

print("Data collator configured.")
print(f"Output dir: {instruction_output_dir}")
print(f"Adapter dir: {instruction_adapter_dir}")

Data collator configured.
Output dir: /content/bihar_instruction_lora_output
Adapter dir: /content/bihar_instruction_lora_adapter


In [68]:
# Training arguments for instruction fine-tuning
# Lower learning rate than Stage 1 to avoid catastrophic forgetting
instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,
    num_train_epochs=5,
    max_steps=-1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,       # Lower LR for instruction FT
    warmup_steps=5,
    weight_decay=config.weight_decay,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=25,
    save_total_limit=2,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
    seed=config.seed,
)

print("Instruction training arguments configured.")

Instruction training arguments configured.


---
## Step 31: Build Trainer and Start Instruction Fine-Tuning

In [69]:
# Build Trainer for instruction fine-tuning
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized["train"],
    eval_dataset=instruction_tokenized["validation"],
    data_collator=instruction_data_collator,
    processing_class=instruction_tokenizer,
)

print("Instruction Trainer ready.")
print(f"Train examples: {len(instruction_tokenized['train'])}")
print(f"Validation examples: {len(instruction_tokenized['validation'])}")

Instruction Trainer ready.
Train examples: 15
Validation examples: 3


In [70]:
# ============================================================
# TRAIN - Instruction Fine-Tuning!
# ============================================================
instruction_train_result = instruction_trainer.train()

print("\n" + "=" * 60)
print("INSTRUCTION FINE-TUNING COMPLETE!")
print("=" * 60)
print(f"Train loss: {instruction_train_result.training_loss:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
5,1.892909,1.976835
10,1.575371,1.771488



INSTRUCTION FINE-TUNING COMPLETE!
Train loss: 1.8163


In [71]:
print("Instruction fine-tuning completed.")
print(f"Train loss: {instruction_train_result.training_loss:.4f}")

# Print training log history
print("\nTraining log history:")
for log in instruction_trainer.state.log_history:
    print(log)

Instruction fine-tuning completed.
Train loss: 1.8163

Training log history:
{'loss': 1.9750579595565796, 'grad_norm': 1.3830232620239258, 'learning_rate': 0.0, 'epoch': 0.5333333333333333, 'step': 1}
{'loss': 2.039802312850952, 'grad_norm': 1.540549397468567, 'learning_rate': 2e-05, 'epoch': 1.0, 'step': 2}
{'loss': 1.9549975395202637, 'grad_norm': 1.3625493049621582, 'learning_rate': 4e-05, 'epoch': 1.5333333333333332, 'step': 3}
{'loss': 1.9731464385986328, 'grad_norm': 1.4629335403442383, 'learning_rate': 6e-05, 'epoch': 2.0, 'step': 4}
{'loss': 1.8929089307785034, 'grad_norm': 1.2297004461288452, 'learning_rate': 8e-05, 'epoch': 2.533333333333333, 'step': 5}
{'eval_loss': 1.9768348932266235, 'eval_runtime': 1.5565, 'eval_samples_per_second': 1.927, 'eval_steps_per_second': 1.927, 'epoch': 2.533333333333333, 'step': 5}
{'loss': 1.7915242910385132, 'grad_norm': 1.21021568775177, 'learning_rate': 0.0001, 'epoch': 3.0, 'step': 6}
{'loss': 1.746857762336731, 'grad_norm': 1.095613718032

---
## Step 32: Save Instruction-Tuned LoRA Adapter

In [72]:
# Save instruction-tuned LoRA adapter + tokenizer
instruction_trainer.model.save_pretrained(instruction_adapter_dir)
instruction_tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(f"\nSaved files:")
for f in os.listdir(instruction_adapter_dir):
    size = os.path.getsize(os.path.join(instruction_adapter_dir, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

Instruction-tuned LoRA adapter saved to: /content/bihar_instruction_lora_adapter

Saved files:
  README.md (5.1 KB)
  tokenizer.json (3534.3 KB)
  adapter_model.safetensors (49319.9 KB)
  tokenizer_config.json (0.4 KB)
  adapter_config.json (1.1 KB)


---
## Step 33: Reload Instruction-Tuned Model for Inference

We reload the model to test instruction-following capability.

Now the model should be able to:
- Understand a question about Bihar
- Generate a structured answer (not just continue text)

In [73]:
# Clean up training objects
del instruction_trainer, instruction_model, instruction_base_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Reload merged Stage 1 model + instruction adapter
if use_cuda:
    inst_inference_base = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inst_inference_base = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Load instruction adapter on top
inst_inference_model = PeftModel.from_pretrained(
    inst_inference_base,
    instruction_adapter_dir,
)
inst_inference_model.eval()

print("Instruction-tuned model loaded for inference!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Instruction-tuned model loaded for inference!


---
## Step 34: Instruction-Style Inference

For instruction-tuned models, prompts must follow the Alpaca format:

```text
### Instruction:
{question}

### Response:

```

The model generates text after `### Response:`

In [74]:
def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    """
    Build an Alpaca-style prompt for instruction inference.

    Parameters:
    - instruction: The user's question or instruction
    - input_text: Optional additional context

    Returns:
    - Formatted prompt string ending with "### Response:\n"
    """
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_instruction_response(
    instruction: str,
    input_text: str = "",
    max_new_tokens: int = 150
) -> str:
    """
    Generate a response to an instruction using the instruction-tuned model.
    """
    prompt = build_instruction_prompt(instruction, input_text)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = instruction_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inst_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=instruction_tokenizer.eos_token_id,
            eos_token_id=instruction_tokenizer.eos_token_id,
        )

    return instruction_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [75]:
# ============================================================
# TEST: Instruction-tuned Bihar model
# ============================================================

test_questions = [
    "Explain the historical significance of Bihar as a political entity.",
    "What were the consequences of Bihar's bifurcation in 2000?",
    "Describe the flood problem in Bihar and its economic impact.",
    "What governance reforms improved Bihar's law and order after 2005?",
]

for question in test_questions:
    print("=" * 90)
    print(f"QUESTION: {question}")
    print("-" * 90)
    response = generate_instruction_response(question, max_new_tokens=150)
    print(f"MODEL RESPONSE:\n{response}")
    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: Explain the historical significance of Bihar as a political entity.
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
Explain the historical significance of Bihar as a political entity.

### Response:
Bihar was created in 1906 and became one of India’s 28 states by the 7th Schedule of the Constitution. The creation of Bihar was intended to make it distinct from both Jharkhand and Uttar Pradesh, and was part of a larger effort to create distinctive and independent states for the Indian provinces. Historically, the geography of Bihar has been unique, and its history is often viewed as one of continuous conflict between different peoples who sought to control it. In the nineteenth century, Bihar was largely controlled by the British, but this changed after independence when the state’s economy was dominated by agriculture, with industrialization slowly beginning

QUESTION: What were the consequences of Bihar's bifurcation in 2000?
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
What were the consequences of Bihar's bifurcation in 2000?

### Response:
Bihar was split into two separate states. The state of Bihar was reduced to the current state of Jharkhand and the state of India was reduced to the current state of Uttar Pradesh. This led to a major loss of population from Bihar and led to serious political, social, economic and environmental challenges for Bihar. A number of factors contributed to this decline – poverty, illiteracy, and poor infrastructure are some of them. These led to low levels of economic activity, low tax revenues, limited access to public services, and a high incidence of crime, corruption and poverty. Apart from these, there were other challenges that Bihar had to face. In

QUESTION: Describe the flood problem in Bihar and its economic impact.
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
Describe the flood problem in Bihar and its economic impact.

### Response:
Bihar is one of the most densely populated states in India with 12,000 people per square kilometre; it is also one of the poorest states with a national poverty rate of 46%. The state has long suffered from acute water shortages, with 95% of the state’s surface area being classified as ‘water-scarce’ (Kumar et al. 2013). This has had devastating consequences on agriculture and horticulture in Bihar, with a decline in crop yields by 35-70% over the past decade. In fact, agricultural productivity has fallen so low that many farmers

QUESTION: What governance reforms improved Bihar's law and order after 2005?
------------------------------------------------------------------------------------------
MODEL RESPONSE:
### Instruction:
What governance reforms improved Bihar's law and order after 2005?

### Response:
The key reforms were the introduction of a new police code in 2013, whi

---
## Step 35: Merge Instruction-Tuned LoRA Adapter

We merge the instruction adapter into the merged Stage 1 model to create
a standalone instruction-tuned model for Stage 3.

```text
Merged Stage 1 model + Stage 2 instruction LoRA adapter
   ↓ merge_and_unload()
Standalone instruction-tuned Bihar model (ready for Stage 3)
```

In [76]:
# Clean up inference objects
del inst_inference_model, inst_inference_base
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Merge instruction adapter into merged Stage 1 model
merged_instruction_model_dir = "/content/bihar_instruction_merged_model"
os.makedirs(merged_instruction_model_dir, exist_ok=True)

# Load merged Stage 1 model in float16 for safe merging
merge_inst_base = AutoModelForCausalLM.from_pretrained(
    merged_model_dir,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None,
    trust_remote_code=True,
)

# Load instruction LoRA adapter
merge_inst_model = PeftModel.from_pretrained(merge_inst_base, instruction_adapter_dir)

# Merge adapter weights into model
merged_instruction_model = merge_inst_model.merge_and_unload()

# Save merged instruction-tuned model
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
instruction_tokenizer.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/bihar_instruction_merged_model


In [77]:
# Clean up merge objects
del merge_inst_base, merge_inst_model, merged_instruction_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

print("Stage 2 complete. Ready for Stage 3: Preference Tuning with DPO.")

Stage 2 complete. Ready for Stage 3: Preference Tuning with DPO.


---
---

# Stage 3: Preference Tuning with DPO (Direct Preference Optimization)

## Teaching the Model Which Answer Is Better

In Stage 1, we adapted the model to the Bihar domain using raw text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we use **preference data** with DPO to teach the model which answers are better.

### What is DPO?

| Aspect | Description |
|--------|-------------|
| Full form | Direct Preference Optimization |
| Paper | NeurIPS 2023, Stanford researchers |
| Main idea | Teach the model which answer is better vs weaker |
| Data format | `prompt` + `chosen` (good answer) + `rejected` (weak answer) |
| vs RLHF | DPO is simpler — no separate reward model or PPO needed |
| vs SFT | SFT teaches *how to answer*; DPO teaches *which answer is better* |

### DPO Data Format

```json
{
  "prompt": "### Instruction:\nExplain Bihar's bifurcation in 2000.\n\n### Response:\n",
  "chosen": "The bifurcation of Bihar into Bihar and Jharkhand in November 2000...",
  "rejected": "Bihar was split in 2000."
}
```

### Pipeline

```text
Merged instruction-tuned Bihar model (Stage 1 + Stage 2)
   ↓
Load as base model
   ↓
Add new LoRA adapter for preference tuning
   ↓
Train with DPO on preference data (chosen vs rejected)
   ↓
Save preference-tuned LoRA adapter
   ↓
Reload and test
```

### How the 3 Stages Compare

| Stage | Data Format | Model Learns |
|-------|-------------|--------------|
| Stage 1 (Non-Instruction) | Raw text | Domain language and knowledge |
| Stage 2 (Instruction) | Instruction → Response | How to answer questions |
| Stage 3 (Preference/DPO) | Prompt → Chosen vs Rejected | Which answer is better |

---
## Step 36: Install TRL for DPO Training

TRL (Transformer Reinforcement Learning) provides `DPOTrainer` and `DPOConfig`
for preference tuning.

In [78]:
!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.1 MB/s eta 0:00:00


In [79]:
from trl import DPOTrainer, DPOConfig
print("TRL imported successfully.")

TRL imported successfully.


---
## Step 37: Create Bihar Preference Dataset

DPO requires data with three columns:
- `prompt`: The instruction/question in Alpaca format
- `chosen`: The better, more complete answer
- `rejected`: The weaker, less helpful answer

The model learns to prefer `chosen` over `rejected`.

In production, you would:
1. Generate multiple answers from the model
2. Have human annotators rank them
3. Or use a stronger model (GPT-4) to judge which is better

For this demo, we create synthetic preference pairs.

In [80]:
# ============================================================
# Create Bihar preference dataset for DPO
# ============================================================
# Each entry has: prompt (Alpaca format), chosen (good answer), rejected (weak answer)

bihar_preference_data = [
    {
        "prompt": "### Instruction:\nExplain the historical significance of Bihar as a political entity.\n\n### Response:\n",
        "chosen": "Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region was a center of political power, learning, and cultural development. The ancient university of Nalanda was located in Bihar, and the region saw the birth of Buddhism and Jainism. Pataliputra served as the capital of successive empires for over a millennium.",
        "rejected": "Bihar is a state in India with some history."
    },
    {
        "prompt": "### Instruction:\nWhat was the impact of the Permanent Settlement Act of 1793 on Bihar?\n\n### Response:\n",
        "chosen": "The Permanent Settlement Act of 1793 had devastating consequences for Bihar's rural population. It created a class of zamindars who became intermediaries between the British government and the cultivators. The peasants were subjected to excessive rent extraction, leading to widespread poverty and indebtedness. The system concentrated land ownership in the hands of a few while impoverishing the majority of agricultural workers.",
        "rejected": "The Permanent Settlement was a British law about land."
    },
    {
        "prompt": "### Instruction:\nDescribe the consequences of Bihar's bifurcation in 2000.\n\n### Response:\n",
        "chosen": "The bifurcation of Bihar into Bihar and Jharkhand in November 2000 resulted in Bihar losing approximately 45% of its geographical area along with significant mineral and forest resources that went to Jharkhand. Bihar retained the more densely populated plains but lost its industrial base. The state's revenue declined sharply, and it had to rebuild its economic foundations with a primarily agrarian economy and limited mineral resources.",
        "rejected": "Bihar was split into two states in 2000."
    },
    {
        "prompt": "### Instruction:\nWhat reforms did the Nitish Kumar government implement after 2005?\n\n### Response:\n",
        "chosen": "After the 2005 elections, the Nitish Kumar government focused on governance reforms including improved law and order through the Bihar Special Courts Act, massive road construction across the state, education expansion with programs like Mukhyamantri Balika Cycle Yojana, and women's empowerment through 50% reservation in panchayat seats. These reforms led to a period of accelerated economic growth often called Bihar's turnaround.",
        "rejected": "Nitish Kumar did some reforms after 2005."
    },
    {
        "prompt": "### Instruction:\nExplain Bihar's economic performance relative to the national average.\n\n### Response:\n",
        "chosen": "Bihar's per capita income has historically been significantly below the national average. After bifurcation in 2000, the gap widened further as the state lost its mineral-rich southern region. However, from 2005 onwards, Bihar experienced higher-than-national-average GDP growth rates, with GSDP growth averaging over 10% annually during 2005-2015. Despite this impressive growth rate, the absolute per capita income gap remained substantial due to the extremely low starting base.",
        "rejected": "Bihar is poor compared to other states."
    },
    {
        "prompt": "### Instruction:\nDescribe the agricultural economy of Bihar.\n\n### Response:\n",
        "chosen": "Agriculture is the backbone of Bihar's economy, employing over 70% of the workforce. The state is a major producer of rice, wheat, maize, lentils, and vegetables. The fertile Gangetic plains provide ideal conditions for cultivation, but the sector suffers from fragmented landholdings with average farm sizes among the smallest in India, frequent floods, limited irrigation coverage, and poor market linkages that prevent farmers from getting fair prices.",
        "rejected": "Bihar grows rice and wheat."
    },
    {
        "prompt": "### Instruction:\nExplain the flood problem in Bihar and its economic impact.\n\n### Response:\n",
        "chosen": "Bihar is one of the most flood-prone states in India, with about 73% of its area susceptible to flooding from rivers originating in Nepal. The Kosi, Gandak, and Bagmati rivers cause recurring devastation. Floods destroy crops, livestock, and infrastructure annually, displacing millions of people. The economic cost includes loss of agricultural output, damage to roads and buildings, and long-term setbacks to development in affected districts.",
        "rejected": "Bihar has floods sometimes."
    },
    {
        "prompt": "### Instruction:\nWhat is the significance of Pataliputra in Bihar's history?\n\n### Response:\n",
        "chosen": "Pataliputra, modern-day Patna, served as the capital of successive empires including the Magadha, Nanda, Maurya, and Gupta dynasties. Founded around the 5th century BCE, it was one of the largest cities in the ancient world. Greek ambassador Megasthenes described it as a magnificent city with elaborate fortifications. Pataliputra was a center of trade, learning, and administration for over a millennium, making it one of the longest-serving capital cities in world history.",
        "rejected": "Pataliputra is the old name of Patna."
    },
    {
        "prompt": "### Instruction:\nHow has Bihar's Human Development Index changed over time?\n\n### Response:\n",
        "chosen": "Bihar has consistently ranked among the lowest Indian states on the Human Development Index. Key indicators including life expectancy, literacy rate, and per capita income have been below national averages. However, post-2005 reforms led to measurable improvements in education enrollment, infant mortality rates, and poverty reduction. The HDI improved from 0.367 in 2001 to approximately 0.447 by 2011, though it remained among the lowest nationally.",
        "rejected": "Bihar's HDI is low."
    },
    {
        "prompt": "### Instruction:\nExplain the role of migration in Bihar's economy.\n\n### Response:\n",
        "chosen": "Bihar is one of the largest sources of internal migration in India. Millions of workers migrate to states like Maharashtra, Delhi, Punjab, and Gujarat for employment in construction, manufacturing, and services. Remittances from migrant workers form a significant portion of household income in Bihar. While migration provides economic relief to families, it also reflects the lack of local employment opportunities and contributes to brain drain from the state.",
        "rejected": "People from Bihar go to other states for work."
    },
    {
        "prompt": "### Instruction:\nWhat healthcare challenges exist in Bihar?\n\n### Response:\n",
        "chosen": "Bihar's healthcare system faces severe challenges including a critical shortage of medical professionals with one of the lowest doctor-to-population ratios in India, inadequate primary health centers in rural areas, high infant and maternal mortality rates, poor sanitation infrastructure, and limited access to specialist care. Despite improvements in immunization coverage and institutional delivery rates, health outcomes remain significantly below national averages.",
        "rejected": "Bihar has bad hospitals."
    },
    {
        "prompt": "### Instruction:\nDescribe Bihar's poverty statistics and trends.\n\n### Response:\n",
        "chosen": "Bihar has historically had one of the highest poverty rates in India. According to the Tendulkar methodology, about 53.5% of Bihar's population lived below the poverty line in 2004-05. This declined to approximately 33.7% by 2011-12, representing significant improvement but still remaining well above the national average of 21.9%. Rural poverty has been particularly acute, driven by landlessness, low agricultural productivity, and limited non-farm employment opportunities.",
        "rejected": "Many people in Bihar are poor."
    },
    {
        "prompt": "### Instruction:\nWhat transportation challenges does Bihar face?\n\n### Response:\n",
        "chosen": "Bihar faces significant transportation challenges including inadequate road connectivity in rural areas, limited railway network coverage relative to population, frequent disruption of road and rail links due to flooding, and insufficient bridges across major rivers. The Ganga river divides the state, and until recently only a few bridges connected north and south Bihar. Road quality has improved substantially since 2005 but still lags behind national standards in many districts.",
        "rejected": "Bihar has bad roads."
    },
    {
        "prompt": "### Instruction:\nExplain the governance reforms that improved Bihar's law and order after 2005.\n\n### Response:\n",
        "chosen": "After 2005, the Bihar government implemented several governance reforms to address severe law and order problems. These included establishing fast-track courts for speedy justice, strengthening police forces with better training and equipment, cracking down on the criminal-political nexus, seizing illegally held arms, and enacting the Bihar Special Courts Act. The government also focused on dismantling private armies and reducing kidnapping-for-ransom incidents. These measures significantly improved the investment climate and public confidence.",
        "rejected": "Law and order got better in Bihar after 2005."
    },
    {
        "prompt": "### Instruction:\nWhat role did the Magadha Empire play in Indian history?\n\n### Response:\n",
        "chosen": "The Magadha Empire, centered in present-day Bihar with its capital at Pataliputra, was one of the most powerful kingdoms in ancient India. It gave rise to the Maurya dynasty under Chandragupta Maurya and reached its zenith under Emperor Ashoka, who spread Buddhism across Asia. Magadha was the birthplace of both Buddhism and Jainism. The subsequent Gupta Empire ushered in India's Golden Age of science, mathematics, astronomy, and art.",
        "rejected": "Magadha was an old kingdom in Bihar."
    },
]

print(f"Created {len(bihar_preference_data)} preference pairs for DPO training.")
print(f"\nSample entry:")
print(json.dumps(bihar_preference_data[0], indent=2))

Created 15 preference pairs for DPO training.

Sample entry:
{
  "prompt": "### Instruction:\nExplain the historical significance of Bihar as a political entity.\n\n### Response:\n",
  "chosen": "Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region was a center of political power, learning, and cultural development. The ancient university of Nalanda was located in Bihar, and the region saw the birth of Buddhism and Jainism. Pataliputra served as the capital of successive empires for over a millennium.",
  "rejected": "Bihar is a state in India with some history."
}


In [81]:
# Save preference data to JSONL for auditability
preference_data_path = os.path.join(config.processed_data_dir, "bihar_preference_dataset.jsonl")

with open(preference_data_path, "w", encoding="utf-8") as f:
    for record in bihar_preference_data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Preference dataset saved to: {preference_data_path}")

Preference dataset saved to: /content/bihar_processed_data/bihar_preference_dataset.jsonl


---
## Step 38: Load Preference Dataset + Train/Eval Split

In [82]:
# Load preference dataset
from datasets import load_dataset

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print(f"Preference dataset: {preference_dataset}")
print(f"\nSample entry:")
print(preference_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Preference dataset: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 15
})

Sample entry:
{'prompt': '### Instruction:\nExplain the historical significance of Bihar as a political entity.\n\n### Response:\n', 'chosen': 'Bihar has been a distinct political entity since ancient times. It was the seat of the Magadha Empire, which gave rise to the Maurya and Gupta dynasties. The region was a center of political power, learning, and cultural development. The ancient university of Nalanda was located in Bihar, and the region saw the birth of Buddhism and Jainism. Pataliputra served as the capital of successive empires for over a millennium.', 'rejected': 'Bihar is a state in India with some history.'}


In [83]:
# Train/validation split
preference_split = preference_dataset.train_test_split(
    test_size=0.15,
    seed=config.seed
)

preference_datasets = DatasetDict({
    "train": preference_split["train"],
    "validation": preference_split["test"],
})

print(preference_datasets)
print(f"\nTrain rows: {len(preference_datasets['train'])}")
print(f"Validation rows: {len(preference_datasets['validation'])}")

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 12
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 3
    })
})

Train rows: 12
Validation rows: 3


---
## Step 39: Load Merged Instruction Model + New LoRA Adapter for DPO

For DPO, we use the **merged instruction-tuned model** as the base.

Then we attach a **fresh LoRA adapter** for preference tuning.

```text
Merged instruction-tuned Bihar model (Stage 1 + Stage 2 baked in)
   + New LoRA adapter (learns preference behavior)
   = Stage 3 trainable model
```

In [95]:
# Clear memory
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
print(torch.cuda.get_device_name(0))
# Load merged instruction-tuned model
if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
    preference_base_model = prepare_model_for_kbit_training(preference_base_model)
else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

print("Merged instruction-tuned model loaded for DPO preference tuning.")

Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Merged instruction-tuned model loaded for DPO preference tuning.


In [96]:
# Create a new LoRA adapter for preference tuning
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

preference_model = get_peft_model(preference_base_model, preference_lora_config)
preference_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---
## Step 40: Configure DPO Training

Key DPO parameters:
- **beta (β)**: Controls how strongly the model is pushed toward chosen answers. Default: 0.1
- **max_length**: Maximum total sequence length (prompt + response)
- **max_prompt_length**: Maximum length for the prompt portion

| DPO Metric | What It Means |
|------------|---------------|
| rewards/chosen | DPO implicit reward for preferred answer (should be higher) |
| rewards/rejected | DPO implicit reward for rejected answer (should be lower) |
| rewards/margins | Difference between chosen and rejected (positive is good) |
| rewards/accuracies | How often model ranks chosen above rejected |
| logps/chosen | Log probability of chosen answer (less negative = more likely) |
| logps/rejected | Log probability of rejected answer (should be more negative) |

In [100]:
# Output directories for DPO preference tuning
preference_output_dir = "/content/bihar_preference_dpo_output"
preference_adapter_dir = "/content/bihar_preference_dpo_lora_adapter"

os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

# DPO training configuration
dpo_training_args = DPOConfig(
    output_dir=preference_output_dir,

    # Training duration
    num_train_epochs=3,
    max_steps=-1,

    # Batch settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings
    learning_rate=5e-5,        # Lower LR for preference tuning
    warmup_steps=2,
    weight_decay=0.01,

    # Logging and evaluation
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=5,

    # Checkpoint saving
    save_steps=25,
    save_total_limit=2,

    # Precision settings
    fp16=False,
    bf16=False,

    # Disable external logging
    report_to="none",

    # Keep required columns
    remove_unused_columns=False,

    # DPO hyperparameter
    # beta controls how strongly the model prefers chosen over rejected
    beta=0.1,
)

print("DPO training arguments configured.")

DPO training arguments configured.


---
## Step 41: Build DPO Trainer and Start Preference Tuning

The DPO Trainer takes:
- **model**: The preference model (with LoRA adapter)
- **ref_model**: Reference model (set to `None` — TRL uses the model's initial state internally)
- **train_dataset**: Preference pairs (prompt, chosen, rejected)
- **processing_class**: Tokenizer for encoding

In [101]:
# Reload tokenizer for DPO
dpo_tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
if dpo_tokenizer.pad_token is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

# Build DPO Trainer
dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,   # TRL internally handles reference behavior
    args=dpo_training_args,
    train_dataset=preference_datasets["train"],
    eval_dataset=preference_datasets["validation"],
    processing_class=dpo_tokenizer,
)

print("DPO Trainer ready.")
print(f"Train preference pairs: {len(preference_datasets['train'])}")
print(f"Validation preference pairs: {len(preference_datasets['validation'])}")

DPO Trainer ready.
Train preference pairs: 12
Validation preference pairs: 3


In [102]:
# ============================================================
# TRAIN DPO! (Fixed for T4 GradScaler Conflict)
# ============================================================

# 1. Force the model and trainer to use float16 but disable the GradScaler
# setting fp16=False in Trainer prevents the 'unscale' error on T4
# while the model weights are already handled via 4-bit/float16 compute.
dpo_trainer.args.fp16 = False
dpo_trainer.args.bf16 = False

# 2. Start training
dpo_train_result = dpo_trainer.train()

print("\n" + "=" * 60)
print("DPO PREFERENCE TUNING COMPLETE!")
print("=" * 60)
print(f"DPO train result: {dpo_train_result}")

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
5,0.393124,0.556561,1.696462,5684.000000,-3.092000,-2.978448,0.618485,0.240626,-0.054628,1.000000,0.295254,-169.428421,-29.319298
6,0.377534,0.544192,1.694451,6447.000000,-3.090631,-2.976438,0.618485,0.266007,-0.058501,1.000000,0.324508,-169.174611,-29.358028



DPO PREFERENCE TUNING COMPLETE!
DPO train result: TrainOutput(global_step=6, training_loss=0.5485778599977493, metrics={'train_runtime': 43.6708, 'train_samples_per_second': 0.824, 'train_steps_per_second': 0.137, 'total_flos': 63444738465792.0, 'train_loss': 0.5485778599977493, 'epoch': 3.0})


---
## Step 42: Save DPO Preference-Tuned LoRA Adapter

In [103]:
# Save DPO preference-tuned LoRA adapter + tokenizer
dpo_trainer.model.save_pretrained(preference_adapter_dir)
dpo_tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
print(f"\nSaved files:")
for f in os.listdir(preference_adapter_dir):
    size = os.path.getsize(os.path.join(preference_adapter_dir, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

Preference-tuned LoRA adapter saved to: /content/bihar_preference_dpo_lora_adapter

Saved files:
  ref (4.0 KB)
  README.md (5.1 KB)
  tokenizer.json (3534.1 KB)
  adapter_model.safetensors (24680.0 KB)
  tokenizer_config.json (0.4 KB)
  adapter_config.json (1.1 KB)


---
## Step 43: Reload Preference-Tuned Model for Inference

We reload the full pipeline:
1. Merged instruction-tuned model (base)
2. DPO preference adapter (on top)

The model should now give more complete, well-structured answers
compared to the Stage 2 instruction-tuned model.

In [104]:
# Clean up DPO training objects
del dpo_trainer, preference_model, preference_base_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Reload merged instruction model + DPO adapter
if use_cuda:
    pref_inference_base = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    pref_inference_base = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Load DPO preference adapter on top
pref_inference_model = PeftModel.from_pretrained(
    pref_inference_base,
    preference_adapter_dir,
)
pref_inference_model.eval()

print("Preference-tuned model loaded for inference!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded for inference!


---
## Step 44: Preference-Tuned Inference

The prompt format remains the same Alpaca-style as Stage 2.

The difference is in the quality of responses — the model should now prefer
more complete, well-structured answers over vague or incomplete ones.

In [105]:
def build_preference_prompt(instruction: str, input_text: str = "") -> str:
    """Build Alpaca-style prompt for preference-tuned inference."""
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_preference_response(
    instruction: str,
    input_text: str = "",
    max_new_tokens: int = 150
) -> str:
    """
    Generate a response using the preference-tuned model.
    """
    prompt = build_preference_prompt(instruction, input_text)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = dpo_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = pref_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=dpo_tokenizer.eos_token_id,
            eos_token_id=dpo_tokenizer.eos_token_id,
        )

    return dpo_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [106]:
# ============================================================
# TEST: Preference-tuned Bihar model
# ============================================================

preference_test_questions = [
    "Explain the historical significance of Bihar as a political entity.",
    "Describe the consequences of Bihar's bifurcation in 2000.",
    "What healthcare challenges exist in Bihar?",
    "Explain the governance reforms that improved Bihar's law and order after 2005.",
    "What role did the Magadha Empire play in Indian history?",
]

for question in preference_test_questions:
    print("=" * 90)
    print(f"QUESTION: {question}")
    print("-" * 90)
    response = generate_preference_response(question, max_new_tokens=150)
    print(f"MODEL RESPONSE:\n{response}")
    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: Explain the historical significance of Bihar as a political entity.
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
Explain the historical significance of Bihar as a political entity.

### Response:
Bihar has been historically known as 'Sumeru' (meaning the land of the Sumerians). The first recorded reference to this land is in the 13th century BC, when King Bimbisara ruled and built the city of Pataliputra. The name Pataliputra was later shortened to Patna, and over time, this region came under the control of different kingdoms such as the Magadha Empire, the Gupta Empire, the Mamluk Sultanate and finally the Delhi Sultanate. In the early 17th century AD, Bihar was divided between the Mughal Emperor, Shah Jahan, and his son, Aurangzeb,

QUESTION: Describe the consequences of Bihar's bifurcation in 2000.
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
Describe the consequences of Bihar's bifurcation in 2000.

### Response:
The consequences of bifurcating Bihar were many and varied. A number of states with comparatively similar economic structures emerged, but all had very different histories. The economy of Bihar was largely agricultural, and much of its economy relied on rice production, which was an important part of the economy during the pre-Independence period but fell out of favor after the Second World War; this gave rise to the question of how to diversify the economy. While the state had some large reserves of natural resources such as coal and natural gas, it was also dependent upon the services sector, especially the IT industry, for employment. There is little doubt that bifurcating Bihar would have been

QUESTION: What healthcare challenges exist in Bihar?
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
What healthcare challenges exist in Bihar?

### Response:
The main challenge in healthcare delivery is access. While Bihar has made strides in improving health infrastructure, there are still many districts that have no medical facility available to them and a significant proportion of people in Bihar live in rural areas where access to health care is often limited. To address this, the government of Bihar has undertaken various initiatives aimed at improving healthcare delivery, including:

- Increasing the number of medical officers (MOs) in the state’s district hospitals;
- Creating a system for monitoring hospitals with the help of MOs and other staff; and
- Building more public health centers across the state to provide primary healthcare services. These

QUESTION: Explain the governance reforms that improved Bihar's law and order after 2005.
------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL RESPONSE:
### Instruction:
Explain the governance reforms that improved Bihar's law and order after 2005.

### Response:
The government under Nitish Kumar’s leadership carried out a range of measures to improve law and order, starting with rehabilitation of Maoist terrorists (see point 10) and the removal of bureaucratic hurdles for police. A number of changes were made in police administration, including the introduction of a new cadre structure and a merit-based recruitment system that increased the size of the force and provided more opportunities for promotion. The government also introduced a number of new laws to make it easier to prosecute criminals, such as the Prevention of Corruption Act, 2003 and the Protection of Women from Domestic Violence Act, 2

QUESTION: What role did the Magadha Empire play in Indian history?
------------------------------------------------------------------------------------------
MODEL RESPONSE:
### Instruction:
What role did the Magadha Empir

---
## Step 45: Optional — Merge DPO Preference Adapter Into Base Model

This creates a fully standalone final model with all three stages baked in:

```text
Base TinyLlama
   → Stage 1: Bihar domain knowledge (merged)
   → Stage 2: Instruction following (merged)
   → Stage 3: Preference alignment (merged)
   = Final standalone Bihar model
```

Use this only when you want a standalone model for deployment.

In [107]:
# Clean up inference objects
del pref_inference_model, pref_inference_base
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Final merged model directory
final_merged_model_dir = "/content/bihar_final_preference_merged_model"
os.makedirs(final_merged_model_dir, exist_ok=True)

# Load merged instruction model in float16 for safe merging
base_for_final_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None,
    trust_remote_code=True,
)

# Attach DPO preference adapter
model_with_pref_adapter = PeftModel.from_pretrained(
    base_for_final_merge,
    preference_adapter_dir,
)

# Merge preference adapter into instruction-tuned model
final_merged_model = model_with_pref_adapter.merge_and_unload()

# Save final standalone model and tokenizer
final_merged_model.save_pretrained(final_merged_model_dir)
dpo_tokenizer.save_pretrained(final_merged_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_model_dir}")
print(f"\nSaved files:")
for f in os.listdir(final_merged_model_dir):
    size = os.path.getsize(os.path.join(final_merged_model_dir, f)) / (1024 * 1024)
    print(f"  {f} ({size:.1f} MB)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/bihar_final_preference_merged_model

Saved files:
  tokenizer.json (3.5 MB)
  model.safetensors (2098.2 MB)
  tokenizer_config.json (0.0 MB)
  generation_config.json (0.0 MB)
  config.json (0.0 MB)
